# 06. Rubrics and LLM-as-Judge

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/06_rubrics_and_llm_as_judge.ipynb)

Deterministic checks are powerful, but many prompt tasks are only partly structured. This notebook shows how to move from keyword checks to rubric scoring and then to an open-source judge model built on Qwen.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Learning goals**
- Understand **When to reach for a rubric**
- Understand **Environment setup**
- Understand **Step 1: Build a judged-answer dataset**
- Understand **Step 2: Start with a deterministic baseline**


In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Learning goals

By the end you will be able to:

- design a small scoring rubric from first principles
- compare deterministic keyword scoring with rubric scoring
- build a judge wrapper around Qwen that returns structured JSON
- probe calibration and simple judge biases

## When to reach for a rubric

Rubrics help when multiple answers can be acceptable, but you still care about dimensions such as accuracy, coverage, clarity, and caution. They are more flexible than exact match, but they introduce subjectivity and judge variance.

## Environment setup

This notebook uses the shared Colab setup so you can run a small open-source judge locally in the notebook.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
import re
from IPython.display import Markdown, display

random.seed(11)

def show_table(rows, columns=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return
    if columns is None:
        columns = list(rows[0].keys())
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    display(Markdown("\n".join([header, divider] + body)))

def clamp(value, low, high):
    return max(low, min(high, value))

def mean(values):
    values = list(values)
    return round(sum(values) / len(values), 3) if values else 0.0

## Step 1: Build a judged-answer dataset

Each row contains a question, a candidate answer, the important reference points, and a simple human score that we will treat as our calibration anchor.

In [ ]:
rubric_examples = [
    {
        "id": "e1",
        "system": "concise_teacher",
        "question": "Why can exact match hide true model quality?",
        "reference_points": [
            "different wording can still be correct",
            "format differences can cause false failures",
            "task-appropriate scoring is often better",
        ],
        "bad_phrases": ["exact match is always enough"],
        "answer": "Exact match only rewards one surface form. A model can be correct with different wording or key order, so you need a scorer that matches the task.",
        "human_overall": 5,
    },
    {
        "id": "e2",
        "system": "glib_prompt",
        "question": "Why can exact match hide true model quality?",
        "reference_points": [
            "different wording can still be correct",
            "format differences can cause false failures",
            "task-appropriate scoring is often better",
        ],
        "bad_phrases": ["exact match is always enough"],
        "answer": "Exact match is bad because language models are creative.",
        "human_overall": 2,
    },
    {
        "id": "e3",
        "system": "structured_prompt",
        "question": "Why should judge prompts include an explicit rubric?",
        "reference_points": [
            "rubrics make criteria explicit",
            "rubrics improve consistency",
            "rubrics reduce vague preferences",
        ],
        "bad_phrases": ["the judge should use any personal preference"],
        "answer": "A rubric tells the judge what matters. That improves consistency across examples and makes the decision easier to audit later.",
        "human_overall": 5,
    },
    {
        "id": "e4",
        "system": "shortcut_prompt",
        "question": "Why should judge prompts include an explicit rubric?",
        "reference_points": [
            "rubrics make criteria explicit",
            "rubrics improve consistency",
            "rubrics reduce vague preferences",
        ],
        "bad_phrases": ["the judge should use any personal preference"],
        "answer": "You mainly need a rubric so the answer looks longer and more academic.",
        "human_overall": 1,
    },
    {
        "id": "e5",
        "system": "concise_teacher",
        "question": "What does temperature control during generation?",
        "reference_points": [
            "temperature changes randomness or diversity",
            "lower temperature is more deterministic",
            "higher temperature can increase variance",
        ],
        "bad_phrases": ["temperature changes model weights"],
        "answer": "Temperature changes how random token selection is. Lower values make generations steadier, while higher values usually create more variation.",
        "human_overall": 5,
    },
    {
        "id": "e6",
        "system": "confident_wrong",
        "question": "What does temperature control during generation?",
        "reference_points": [
            "temperature changes randomness or diversity",
            "lower temperature is more deterministic",
            "higher temperature can increase variance",
        ],
        "bad_phrases": ["temperature changes model weights"],
        "answer": "Temperature rewrites the model's internal weights while it answers, which makes it either smarter or weaker.",
        "human_overall": 1,
    },
    {
        "id": "e7",
        "system": "structured_prompt",
        "question": "Why should we calibrate an LLM judge before trusting it?",
        "reference_points": [
            "compare judge outputs against known examples",
            "check for systematic over or under scoring",
            "look for bias or inconsistency",
        ],
        "bad_phrases": ["judge outputs never drift"],
        "answer": "Calibration means checking the judge on examples with known expectations. That helps reveal systematic inflation, harshness, or bias before you use the judge broadly.",
        "human_overall": 5,
    },
    {
        "id": "e8",
        "system": "keyword_stuffer",
        "question": "Why should we calibrate an LLM judge before trusting it?",
        "reference_points": [
            "compare judge outputs against known examples",
            "check for systematic over or under scoring",
            "look for bias or inconsistency",
        ],
        "bad_phrases": ["judge outputs never drift"],
        "answer": "Calibration is about known examples, systematic scoring, bias, consistency, and drift. Those words are all important.",
        "human_overall": 2,
    },
]

show_table(
    [
        {
            "id": row["id"],
            "system": row["system"],
            "question": row["question"][:48] + "...",
            "human_overall": row["human_overall"],
        }
        for row in rubric_examples
    ]
)

## Step 2: Start with a deterministic baseline

The simplest baseline is a keyword or phrase-coverage score. This is useful as a sanity check, but it cannot tell good concise answers from shallow keyword stuffing.

In [ ]:
def keyword_score(example):
    answer = example["answer"].lower()
    hits = sum(1 for point in example["reference_points"] if any(token in answer for token in point.lower().split()))
    return round(5 * hits / len(example["reference_points"]), 2)

baseline_rows = []
for example in rubric_examples:
    baseline_rows.append(
        {
            "id": example["id"],
            "system": example["system"],
            "keyword_score": keyword_score(example),
            "human_overall": example["human_overall"],
        }
    )

show_table(baseline_rows, columns=["id", "system", "keyword_score", "human_overall"])

## Step 3: Make the rubric explicit

We will score answers on three dimensions:

- accuracy: does the answer avoid false claims?
- coverage: does it address the important ideas?
- clarity: is it direct and easy to use?

The total score ranges from 0 to 5.

In [ ]:
RUBRIC_TEXT = """
Score each answer on these dimensions.

Accuracy: 0 to 2
- 2: factual and careful
- 1: partly right or too vague
- 0: materially wrong

Coverage: 0 to 2
- 2: includes most important points
- 1: includes some important points
- 0: misses the key ideas

Clarity: 0 to 1
- 1: easy to follow and directly useful
- 0: confusing, padded, or badly organized
""".strip()

print(RUBRIC_TEXT)

## Step 4: Build a transparent rubric proxy

Before asking a model to judge, it is useful to write down a rough mechanical approximation. That forces your criteria to become concrete.

In [ ]:
def rubric_proxy(example):
    answer = example["answer"].lower()
    point_hits = 0
    for point in example["reference_points"]:
        tokens = [token for token in re.findall(r"[a-z]+", point.lower()) if len(token) > 3]
        if any(token in answer for token in tokens):
            point_hits += 1

    bad_hits = sum(1 for phrase in example["bad_phrases"] if phrase.lower() in answer)
    accuracy = 2 - bad_hits
    if "always" in answer and "exact match" in answer:
        accuracy -= 1
    coverage = round(2 * point_hits / len(example["reference_points"]))
    word_count = len(re.findall(r"[a-zA-Z]+", example["answer"]))
    clarity = 1 if 10 <= word_count <= 45 else 0

    accuracy = clamp(accuracy, 0, 2)
    coverage = clamp(coverage, 0, 2)
    overall = accuracy + coverage + clarity

    return {
        "accuracy": accuracy,
        "coverage": coverage,
        "clarity": clarity,
        "overall": overall,
    }

proxy_rows = []
for example in rubric_examples:
    proxy = rubric_proxy(example)
    proxy_rows.append(
        {
            "id": example["id"],
            "system": example["system"],
            "proxy_overall": proxy["overall"],
            "human_overall": example["human_overall"],
        }
    )

show_table(proxy_rows, columns=["id", "system", "proxy_overall", "human_overall"])

## Step 5: Create a Qwen judge prompt

Judge prompts work better when they are explicit, bounded, and machine-readable. We will ask for JSON so downstream analysis stays transparent.

In [ ]:
def build_judge_prompt(example):
    reference_points = "\n".join("- " + point for point in example["reference_points"])
    bad_phrases = "\n".join("- " + phrase for phrase in example["bad_phrases"]) or "- None"
    return f"""
You are grading one answer to a short instructional question.

Question:
{example["question"]}

Reference points:
{reference_points}

Known bad claims to watch for:
{bad_phrases}

Candidate answer:
{example["answer"]}

Use this rubric:
{RUBRIC_TEXT}

Return JSON only with keys:
accuracy, coverage, clarity, overall, verdict, evidence

Rules:
- overall must equal accuracy + coverage + clarity
- evidence must be a short list of concrete observations
- do not include markdown fences
""".strip()

print(build_judge_prompt(rubric_examples[0]))

## Step 6: Wrap Qwen as a judge

The wrapper below uses the shared generate helper from the setup cell. It extracts the first JSON object and normalizes the result so the rest of the notebook can work with structured records.

In [ ]:
def extract_first_json(text):
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"No JSON object found in judge output: {text}")
    return match.group(0)

def coerce_judge_record(record):
    accuracy = int(clamp(int(record.get("accuracy", 0)), 0, 2))
    coverage = int(clamp(int(record.get("coverage", 0)), 0, 2))
    clarity = int(clamp(int(record.get("clarity", 0)), 0, 1))
    overall = int(clamp(int(record.get("overall", accuracy + coverage + clarity)), 0, 5))
    return {
        "accuracy": accuracy,
        "coverage": coverage,
        "clarity": clarity,
        "overall": overall,
        "verdict": str(record.get("verdict", "")).strip(),
        "evidence": record.get("evidence", []),
    }

def judge_with_qwen(example):
    raw_text = generate(
        [
            {"role": "system", "content": "You are a careful evaluation judge. Return JSON only."},
            {"role": "user", "content": build_judge_prompt(example)},
        ],
        max_new_tokens=260,
        temperature=0.0,
        do_sample=False,
    )
    parsed = json.loads(extract_first_json(raw_text))
    record = coerce_judge_record(parsed)
    record["raw_text"] = raw_text
    return record

## Step 7: Judge the dataset

The dataset is intentionally small so the experiment stays runnable on a Colab T4. You can expand it once the workflow is stable.

In [ ]:
judge_rows = []
for example in rubric_examples:
    judge = judge_with_qwen(example)
    proxy = rubric_proxy(example)
    judge_rows.append(
        {
            "id": example["id"],
            "system": example["system"],
            "human": example["human_overall"],
            "keyword": keyword_score(example),
            "proxy": proxy["overall"],
            "judge": judge["overall"],
            "verdict": judge["verdict"][:55],
        }
    )

show_table(judge_rows, columns=["id", "system", "human", "keyword", "proxy", "judge", "verdict"])

## Step 8: Compare deterministic scoring and rubric scoring

This summary shows why rubric-driven judgment exists: keyword baselines can miss concise correctness and over-reward shallow phrase matching.

In [ ]:
def mae(rows, predicted_key):
    return round(statistics.mean(abs(row[predicted_key] - row["human"]) for row in rows), 3)

comparison_summary = [
    {"method": "keyword", "mae_vs_human": mae(judge_rows, "keyword")},
    {"method": "rubric_proxy", "mae_vs_human": mae(judge_rows, "proxy")},
    {"method": "qwen_judge", "mae_vs_human": mae(judge_rows, "judge")},
]

show_table(comparison_summary)

## Step 9: Inspect calibration by system type

Calibration is not just an average score. A judge can be fine overall and still systematically favor certain answer styles.

In [ ]:
system_rows = []
for system_name in sorted({row["system"] for row in judge_rows}):
    subset = [row for row in judge_rows if row["system"] == system_name]
    system_rows.append(
        {
            "system": system_name,
            "mean_human": round(statistics.mean(row["human"] for row in subset), 2),
            "mean_judge": round(statistics.mean(row["judge"] for row in subset), 2),
            "judge_minus_human": round(
                statistics.mean(row["judge"] - row["human"] for row in subset), 2
            ),
        }
    )

show_table(system_rows)

## Step 10: Stress-test simple judge biases

A classic failure mode is rewarding style over substance. We will judge the same core answer wrapped in different surface presentations.

In [ ]:
bias_probe = {
    "question": "Why should we calibrate an LLM judge before trusting it?",
    "reference_points": [
        "compare judge outputs against known examples",
        "check for systematic over or under scoring",
        "look for bias or inconsistency",
    ],
    "bad_phrases": ["judge outputs never drift"],
}

core_answer = "Calibration means checking the judge on known examples so you can spot harshness, inflation, and bias before relying on it."

bias_variants = [
    {
        **bias_probe,
        "id": "bias_plain",
        "answer": core_answer,
    },
    {
        **bias_probe,
        "id": "bias_long",
        "answer": core_answer + " It also helps teams build trust, communicate process clearly, and document evaluation practice in a polished way.",
    },
    {
        **bias_probe,
        "id": "bias_authority",
        "answer": "Senior evaluator answer: " + core_answer,
    },
]

bias_rows = []
for example in bias_variants:
    judged = judge_with_qwen(example)
    bias_rows.append(
        {
            "id": example["id"],
            "judge_score": judged["overall"],
            "verdict": judged["verdict"][:60],
        }
    )

show_table(bias_rows)

## Useful takeaways

Rubric-based evaluation is helpful when:

- many answers can be acceptable
- you need graded feedback instead of binary pass or fail
- you care about dimensions like coverage and clarity
- deterministic scoring is obviously underfitting the task

## Important limits

LLM judges introduce their own failure modes:

- bias toward length, style, or confident tone
- drift when prompts change
- inconsistency on borderline examples
- hidden dependence on wording of the judge prompt itself

## Suggested extensions

Good next steps:

1. add a small human-labeled calibration set with disagreements
2. run the judge twice with different prompts and compare stability
3. compare Qwen against another open-source judge model on the same examples

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** design an eval dataset for a new use case. Document what changes and why.

**Exercise 2 — Build:** implement a custom metric and compare it to the one in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** run the evaluation on a different model and analyze the differences.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the evals/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [05 Rule Based Prompt Eval](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/05_rule_based_prompt_eval.ipynb) | ➡️ **Next:** [07 Pairwise And Preference Eval](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/07_pairwise_and_preference_eval.ipynb)